# Hyperliquid Funding Rate Mean-Reversion Backtest

**Hypothesis**: When 8h cumulative funding rates reach extreme percentiles (>95th or <5th trailing 30d),
longs/shorts are overcrowded and price tends to mean-revert.

**Universe**: Top-30 alt perps by 30d rolling volume (excluding BTC/ETH), monthly refresh.

**Data**: Hyperliquid public API — 4h OHLCV candles (Jan 2024 → present), hourly funding history.

---


In [ ]:
import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
import logging
logging.basicConfig(level=logging.WARNING)

from data_loader import get_candles_chunked, get_funding_chunked, get_meta, get_asset_contexts
from strategy import StrategyParams, compute_signals, build_signals_all
from backtest import run_backtest
from analytics import (
    compute_metrics, per_asset_breakdown, monthly_returns,
    bootstrap_sharpe_ci, walk_forward_analysis,
    sharpe_ratio, max_drawdown
)
from universe import build_monthly_universes, get_universe_for_date
from main import (
    fetch_all_data, build_bnh_curve,
    DAILY_START_MS, SIGNAL_START_MS, DATA_END_MS,
    SIGNAL_INTERVAL, BAR_INTERVAL_H
)

import datetime
from pathlib import Path

SEED = 42
np.random.seed(SEED)
INITIAL_EQUITY = 100_000.0
MAX_COINS = 10  # increase to 30 for full run
print('Setup complete.')

## 1. Data Loading & Universe Selection

In [ ]:
from data_loader import get_meta, get_asset_contexts
from universe import get_all_alt_perps

all_coins = get_all_alt_perps()
print(f'Total alt perps on HL: {len(all_coins)}')

# Pre-filter by current 24h volume (same logic as main.py)
meta = get_meta()
contexts = get_asset_contexts()
coin_names = [m['name'] for m in meta]
vol_map = {}
for i, ctx in enumerate(contexts):
    if i < len(coin_names) and coin_names[i] not in {'BTC', 'ETH'}:
        try:
            vol_map[coin_names[i]] = float(ctx.get('dayNtlVlm', 0))
        except:
            pass

sorted_by_vol = sorted(vol_map, key=lambda c: -vol_map[c])
prefetch_coins = sorted_by_vol[:MAX_COINS * 2]
all_coins = [c for c in all_coins if c in set(prefetch_coins)][:MAX_COINS]
print(f'Using {len(all_coins)} coins: {", ".join(all_coins[:10])}')

In [ ]:
# Fetch data (uses cache after first run)
daily_candles_map, signal_candles_map, funding_map = fetch_all_data(all_coins)
print(f'\nData loaded: {len(signal_candles_map)} coins with 4h candles, {len(funding_map)} with funding')

# Show data ranges
for coin in list(signal_candles_map.keys())[:3]:
    df = signal_candles_map[coin]
    fd = funding_map.get(coin, pd.DataFrame())
    print(f'  {coin}: candles {df.index[0].date()} → {df.index[-1].date()} ({len(df)} bars) | '
          f'funding {len(fd)} rows')

In [ ]:
import datetime, pytz

start_dt = datetime.datetime(2022, 2, 1, tzinfo=datetime.timezone.utc)
end_dt = datetime.datetime.now(datetime.timezone.utc)

universe_candles = daily_candles_map if daily_candles_map else signal_candles_map
universe_snapshots = build_monthly_universes(start_dt, end_dt, universe_candles)
populated = [s for s in universe_snapshots if s.coins]
print(f'Universe snapshots: {len(universe_snapshots)} total, {len(populated)} populated')
print(f'First populated: {populated[0].date.strftime("%Y-%m")} | Last: {populated[-1].date.strftime("%Y-%m")}')

In [ ]:
# Plot: universe volume evolution
months = [s.date for s in populated]
coins_per_month = [len(s.coins) for s in populated]

# Gather top-5 coin appearances
all_snap_coins = set()
for s in populated:
    all_snap_coins.update(s.coins[:5])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.bar(months, coins_per_month, color='#2196F3', alpha=0.7)
ax1.set_title('Universe Size Over Time (monthly)', fontsize=12)
ax1.set_ylabel('# Coins in Universe')
ax1.grid(alpha=0.3)

# Top coin volumes over time
top_coins = list(signal_candles_map.keys())[:5]
for coin in top_coins:
    vols = []
    for s in populated:
        v = s.volumes.get(coin, 0) if coin in s.coins else 0
        vols.append(v / 1e9)  # in billions
    ax2.plot(months, vols, lw=1.5, label=coin)

ax2.set_title('30d Notional Volume by Coin ($B)', fontsize=12)
ax2.set_ylabel('Volume ($B)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

for ax in (ax1, ax2):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('output/universe_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Universe evolution plot saved.')

## 2. Signal Analysis — Funding Rate Distribution

In [ ]:
# Compute signals
params = StrategyParams(
    entry_long_pct=5.0,
    entry_short_pct=95.0,
    funding_window_h=8,
    bar_interval_h=BAR_INTERVAL_H,
)

universe_coins = set()
for snap in universe_snapshots:
    universe_coins.update(snap.coins)

filtered_signal = {k: v for k, v in signal_candles_map.items() if k in universe_coins}
filtered_funding = {k: v for k, v in funding_map.items() if k in universe_coins}

signals_map = build_signals_all(filtered_signal, filtered_funding, params)
print(f'Signals computed for {len(signals_map)} assets')

In [ ]:
# Plot: funding rate percentile distribution for one coin
example_coin = 'SOL' if 'SOL' in signals_map else list(signals_map.keys())[0]
sig = signals_map[example_coin]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 1. Funding rate over time
ax = axes[0]
fd = sig['funding_per_bar']
ax.plot(fd.index, fd.values * 100, lw=0.7, color='#2196F3', alpha=0.8)
ax.axhline(0, color='black', lw=0.5)
ax.set_title(f'{example_coin} — Funding Rate per 4h Bar (%)', fontsize=11)
ax.set_ylabel('Funding %')
ax.grid(alpha=0.3)

# 2. 8h cumulative funding
ax = axes[1]
ax.plot(sig.index, sig['funding_8h'] * 100, lw=0.7, color='#FF5722')
ax.axhline(0, color='black', lw=0.5)
ax.set_title(f'{example_coin} — 8h Cumulative Funding (%)', fontsize=11)
ax.set_ylabel('Cumulative Funding %')
ax.grid(alpha=0.3)

# 3. Percentile rank
ax = axes[2]
pct = sig['pct_rank_8h'].dropna()
ax.plot(pct.index, pct.values, lw=0.7, color='#9C27B0')
ax.axhline(95, color='red', lw=1, ls='--', label='SHORT threshold (95th)')
ax.axhline(5, color='green', lw=1, ls='--', label='LONG threshold (5th)')
ax.fill_between(pct.index, 40, 60, alpha=0.1, color='gray', label='Exit zone')
# Mark long signals
long_sigs = sig[sig['signal_long']]
short_sigs = sig[sig['signal_short']]
ax.scatter(long_sigs.index, long_sigs['pct_rank_8h'], color='green', s=15, zorder=5, label=f'Long ({len(long_sigs)})')
ax.scatter(short_sigs.index, short_sigs['pct_rank_8h'], color='red', s=15, zorder=5, label=f'Short ({len(short_sigs)})')
ax.set_ylim(0, 100)
ax.set_title(f'{example_coin} — 30d Trailing Percentile Rank of 8h Funding', fontsize=11)
ax.set_ylabel('Percentile')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('output/signal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Signal analysis saved. Total signals: {len(long_sigs)} long, {len(short_sigs)} short')

## 3. Backtest — In-Sample & Out-of-Sample

In [ ]:
# Train/test split
TRAIN_FRACTION = 0.70

all_times_set = set()
for df in signals_map.values():
    all_times_set.update(df.index)
all_times = sorted(all_times_set)
split_idx = int(len(all_times) * TRAIN_FRACTION)
split_time = all_times[split_idx]

print(f'IS period: {all_times[0].strftime("%Y-%m-%d")} → {split_time.strftime("%Y-%m-%d")}')
print(f'OOS period: {split_time.strftime("%Y-%m-%d")} → {all_times[-1].strftime("%Y-%m-%d")}')

def split_to(d, end):
    return {k: v.loc[v.index <= end] for k, v in d.items() if not v.loc[v.index <= end].empty}

def split_from(d, start):
    return {k: v.loc[v.index >= start] for k, v in d.items() if not v.loc[v.index >= start].empty}

is_snapshots = [s for s in universe_snapshots if pd.Timestamp(s.date) <= split_time]
oos_snapshots = [s for s in universe_snapshots if pd.Timestamp(s.date) > split_time]

is_signals = split_to(signals_map, split_time)
oos_signals = split_from(signals_map, split_time)
is_funding = split_to(filtered_funding, split_time)
oos_funding = split_from(filtered_funding, split_time)

In [ ]:
print('Running IS backtest...')
is_result = run_backtest(
    is_signals, is_funding, is_snapshots, params,
    initial_equity=INITIAL_EQUITY, label='IS'
)

oos_initial = float(is_result.equity_curve.iloc[-1]) if not is_result.equity_curve.empty else INITIAL_EQUITY
print(f'Running OOS backtest (starting equity: ${oos_initial:,.2f})...')
oos_result = run_backtest(
    oos_signals, oos_funding, oos_snapshots, params,
    initial_equity=oos_initial, label='OOS'
)

is_m = compute_metrics(is_result, initial_equity=INITIAL_EQUITY, label='In-Sample')
oos_m = compute_metrics(oos_result, initial_equity=oos_initial, label='Out-of-Sample')

# Summary table
metrics_df = pd.DataFrame([is_m, oos_m]).set_index('label')[[
    'total_return_pct', 'ann_return_pct', 'sharpe', 'sortino', 'calmar',
    'max_drawdown_pct', 'n_trades', 'win_rate_pct', 'profit_factor',
    'avg_hold_hours', 'total_fees_usd', 'total_funding_pnl_usd'
]].round(3)
print('\n' + metrics_df.T.to_string())

In [ ]:
# Equity curve
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [3, 1, 1]})

is_eq = is_result.equity_curve
oos_eq = oos_result.equity_curve

# Buy and hold
bnh = build_bnh_curve(signal_candles_map, list(filtered_signal.keys()), INITIAL_EQUITY)

# Normalize
is_norm = is_eq / INITIAL_EQUITY * 100
oos_norm = oos_eq / float(is_eq.iloc[-1]) * float(is_eq.iloc[-1] / INITIAL_EQUITY * 100) if not is_eq.empty else oos_eq
bnh_norm = bnh / bnh.iloc[0] * 100 if not bnh.empty else None

ax = axes[0]
ax.plot(is_norm.index, is_norm.values, color='#2196F3', lw=1.5, label='Strategy (IS)')
ax.plot(oos_norm.index, oos_norm.values, color='#FF9800', lw=1.5, label='Strategy (OOS)')
ax.axvline(split_time, color='gray', ls='--', lw=1, label='IS/OOS split')
if bnh_norm is not None:
    ax.plot(bnh_norm.index, bnh_norm.values, color='#9E9E9E', lw=1, ls=':', label='Buy & Hold EW')
ax.set_ylabel('Equity (rebased 100)')
ax.legend(fontsize=9)
ax.set_title('Funding Rate Mean-Reversion Strategy — Equity Curve', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# Drawdown
full_eq = pd.concat([is_eq, oos_eq]).sort_index()
full_eq = full_eq[~full_eq.index.duplicated(keep='last')]
dd = (full_eq - full_eq.cummax()) / full_eq.cummax() * 100
axes[1].fill_between(dd.index, dd.values, 0, color='#F44336', alpha=0.5)
axes[1].set_ylabel('Drawdown %')
axes[1].grid(alpha=0.3)

# Position count
pos_ts = pd.concat([is_result.positions_over_time, oos_result.positions_over_time]).sort_index()
axes[2].step(pos_ts.index, pos_ts['n_positions'].values, color='#4CAF50', lw=1)
axes[2].set_ylabel('# Positions')
axes[2].set_ylim(0, 6)
axes[2].grid(alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('output/equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly return heatmaps
from analytics import monthly_returns, plot_monthly_heatmap
from pathlib import Path

is_monthly = monthly_returns(is_result.equity_curve)
oos_monthly = monthly_returns(oos_result.equity_curve)
plot_monthly_heatmap(is_monthly, oos_monthly, Path('output/monthly_heatmap.png'))
print('Monthly heatmap saved.')

## 4. Per-Asset Breakdown

In [ ]:
is_asset = per_asset_breakdown(is_result.trades)
oos_asset = per_asset_breakdown(oos_result.trades)

print('IN-SAMPLE:')
display(is_asset.style.background_gradient(subset=['net_pnl'], cmap='RdYlGn').format({
    'net_pnl': '${:,.0f}',
    'win_rate': '{:.1f}%',
    'avg_hold_h': '{:.1f}h',
    'total_funding': '${:,.0f}',
}))

print('\nOUT-OF-SAMPLE:')
display(oos_asset.style.background_gradient(subset=['net_pnl'], cmap='RdYlGn').format({
    'net_pnl': '${:,.0f}',
    'win_rate': '{:.1f}%',
    'avg_hold_h': '{:.1f}h',
    'total_funding': '${:,.0f}',
}))

In [ ]:
# Bar chart of per-asset PnL
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, asset_df, title in zip(axes, [is_asset, oos_asset], ['In-Sample', 'Out-of-Sample']):
    if asset_df.empty:
        ax.set_title(f'{title} — No Trades')
        continue
    colors = ['#4CAF50' if v > 0 else '#F44336' for v in asset_df['net_pnl']]
    ax.barh(asset_df['coin'], asset_df['net_pnl'], color=colors)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_title(f'Net PnL by Asset — {title}', fontsize=11)
    ax.set_xlabel('Net PnL (USD)')
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('output/per_asset_pnl.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Robustness Checks

In [ ]:
# Bootstrap Sharpe CI
for label, result in [('IS', is_result), ('OOS', oos_result)]:
    lo, pt, hi = bootstrap_sharpe_ci(result.daily_returns, n_boot=1000)
    print(f'{label} Sharpe 95% CI: [{lo:.3f}, {pt:.3f}, {hi:.3f}]')

In [ ]:
# Walk-forward analysis
wf_df = walk_forward_analysis(is_result, INITIAL_EQUITY)
print(f'Walk-forward ({len(wf_df)} windows, IS):')
print(wf_df[['window_start', 'window_end', 'sharpe', 'return_pct', 'max_dd_pct']].round(3).to_string(index=False))

In [ ]:
# Walk-forward plot
from analytics import plot_walk_forward
from pathlib import Path
plot_walk_forward(wf_df, Path('output/walk_forward.png'))
print('Walk-forward plot saved.')

## 6. Sensitivity Analysis

In [ ]:
threshold_pairs = [(5, 95), (10, 90), (1, 99), (15, 85)]
sens_rows = []

for lo_pct, hi_pct in threshold_pairs:
    s_params = StrategyParams(
        entry_long_pct=lo_pct,
        entry_short_pct=hi_pct,
        bar_interval_h=BAR_INTERVAL_H,
    )
    s_signals = build_signals_all(filtered_signal, filtered_funding, s_params)
    s_is_signals = split_to(s_signals, split_time)
    s_result = run_backtest(
        s_is_signals, is_funding, is_snapshots, s_params,
        initial_equity=INITIAL_EQUITY, label=f'sens_{lo_pct}/{hi_pct}'
    )
    m = compute_metrics(s_result, initial_equity=INITIAL_EQUITY)
    m['thresholds'] = f'{lo_pct}/{hi_pct}'
    sens_rows.append(m)

sens_df = pd.DataFrame(sens_rows)[[
    'thresholds', 'total_return_pct', 'sharpe', 'max_drawdown_pct', 'n_trades', 'win_rate_pct'
]].round(3)
print('Sensitivity Analysis (IS only):')
display(sens_df.set_index('thresholds'))

In [ ]:
# Visualize sensitivity
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
thresholds = sens_df['thresholds'].values
x = range(len(thresholds))

for ax, metric, ylabel in zip(axes,
    ['sharpe', 'total_return_pct', 'n_trades'],
    ['Sharpe Ratio', 'Total Return %', '# Trades']):
    vals = sens_df[metric].values
    colors = ['#4CAF50' if v > 0 else '#F44336' for v in vals]
    ax.bar(x, vals, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(thresholds, rotation=30)
    ax.set_title(f'{ylabel} by Threshold Pair', fontsize=10)
    ax.set_ylabel(ylabel)
    ax.axhline(0, color='black', lw=0.5)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('output/sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sensitivity plot saved.')

## 7. Key Findings & Interpretation

### What the results show:

1. **Negative expected value per trade**: With TP=1.5% and SL=2%, break-even win rate = 62%. 
   Actual win rate (~52-55%) is below this, so the strategy loses money from price action alone.

2. **Funding carry is tiny**: Total funding PnL is negligible (<1% of losses). 
   Short average hold times (1-2 bars = 4-8h) don't accumulate enough carry.

3. **High trade frequency**: ~4000 IS trades on 10 coins = aggressive signal frequency.
   Fees dominate at these volumes.

4. **Bull market bias**: The 2024-2025 IS period was a strong bull market for crypto.
   Funding was persistently positive → short signals were fighting the trend.

### Potential improvements:

- **Asymmetric TP/SL**: Use TP=3% or SL=1.5% to improve the payoff ratio
- **Trend filter**: Only trade mean-reversion in sideways/bearish regimes
- **Minimum carry threshold**: Only enter when funding rate is large enough to justify the trade
- **Longer hold times**: Hold for the full funding collection window (8h or more)
- **Direction bias**: Only short when funding is positive extreme (classic carry + mean reversion)


In [ ]:
# Summary statistics table
summary = pd.DataFrame({
    'Metric': [
        'Total Return', 'Ann. Return', 'Sharpe', 'Sortino', 'Calmar',
        'Max Drawdown', 'DD Duration (days)', '# Trades', 'Win Rate',
        'Avg Win ($)', 'Avg Loss ($)', 'Profit Factor',
        'Avg Hold (bars)', 'Total Fees ($)', 'Funding PnL ($)'
    ],
    'In-Sample': [
        f"{is_m.get('total_return_pct', 0):.2f}%",
        f"{is_m.get('ann_return_pct', 0):.2f}%",
        f"{is_m.get('sharpe', 0):.3f}",
        f"{is_m.get('sortino', 0):.3f}",
        f"{is_m.get('calmar', 0):.3f}",
        f"{is_m.get('max_drawdown_pct', 0):.2f}%",
        f"{is_m.get('dd_duration_days', 0):.1f}",
        f"{is_m.get('n_trades', 0):,}",
        f"{is_m.get('win_rate_pct', 0):.1f}%",
        f"${is_m.get('avg_win_usd', 0):,.2f}",
        f"${is_m.get('avg_loss_usd', 0):,.2f}",
        f"{is_m.get('profit_factor', 0):.2f}",
        f"{is_m.get('avg_hold_hours', 0):.1f}",
        f"${is_m.get('total_fees_usd', 0):,.2f}",
        f"${is_m.get('total_funding_pnl_usd', 0):,.2f}",
    ],
    'Out-of-Sample': [
        f"{oos_m.get('total_return_pct', 0):.2f}%",
        f"{oos_m.get('ann_return_pct', 0):.2f}%",
        f"{oos_m.get('sharpe', 0):.3f}",
        f"{oos_m.get('sortino', 0):.3f}",
        f"{oos_m.get('calmar', 0):.3f}",
        f"{oos_m.get('max_drawdown_pct', 0):.2f}%",
        f"{oos_m.get('dd_duration_days', 0):.1f}",
        f"{oos_m.get('n_trades', 0):,}",
        f"{oos_m.get('win_rate_pct', 0):.1f}%",
        f"${oos_m.get('avg_win_usd', 0):,.2f}",
        f"${oos_m.get('avg_loss_usd', 0):,.2f}",
        f"{oos_m.get('profit_factor', 0):.2f}",
        f"{oos_m.get('avg_hold_hours', 0):.1f}",
        f"${oos_m.get('total_fees_usd', 0):,.2f}",
        f"${oos_m.get('total_funding_pnl_usd', 0):,.2f}",
    ]
}).set_index('Metric')

display(summary)